# Reactive travel opportunities from the base graph

All notebook code remains self-contained. The source of city IDs, connection
directions, transport modes, operators and parallel alternatives is the local
`base_graph.pickle` (or `.pickle.zst`) snapshot. No backend-package imports are needed.
Dependencies: Python 3.14, NetworkX, pandas, ipywidgets and IPython.

Run all cells once. Choose a snapshot and home city, then change widgets or
click **Generate new sample**. Only existing graph connections are passed to the
mock APIs across the entire snapshot, without city or regional partitions; schedules and fares remain **synthetic**, not real availability.


In [ ]:
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from html import escape

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

## Domain models

In [ ]:
"""Travel modes owned by the domain rather than a generated wire schema."""

from enum import StrEnum


class TransportType(StrEnum):
    BUSSES = "BUSSES"
    PLANES = "PLANES"
    TRAINS = "TRAINS"
    FERRIES = "FERRIES"

Mode = TransportType

"""Provider-independent, immutable scheduled purchase units and v1 constraints."""

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from decimal import Decimal
import re
import math



def instant(value: datetime) -> datetime:
    if value.tzinfo is None or value.utcoffset() is None:
        raise ValueError("timestamps must include an explicit UTC offset")
    return value.astimezone(timezone.utc)


def identifier(value: str) -> None:
    if not isinstance(value, str) or not value.strip() or value != value.strip():
        raise ValueError(
            "city identifiers must be non-empty without surrounding whitespace"
        )


@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self):
        if (
            not isinstance(self.amount, Decimal)
            or not self.amount.is_finite()
            or self.amount < 0
        ):
            raise ValueError("amount must be a finite non-negative Decimal")
        if not re.fullmatch(r"[A-Z]{3}", self.currency):
            raise ValueError(
                "currency must be an uppercase three-letter ISO currency code"
            )


@dataclass(frozen=True)
class CityAttractiveness:
    city: str
    weight: float

    def __post_init__(self):
        identifier(self.city)
        if (
            type(self.weight) not in (int, float)
            or not math.isfinite(self.weight)
            or self.weight <= 0
        ):
            raise ValueError("attractiveness must be finite and strictly positive")


@dataclass(frozen=True)
class SearchSettings:
    departure_city: str
    departure_date: datetime
    desired_arrival_date: datetime
    time_tolerance: int
    expenses_tolerance: Money
    transport_types: tuple[TransportType, ...]
    mandatory_cities: tuple[str, ...] = ()
    excluded_cities: tuple[str, ...] = ()
    max_trip_duration: int | None = None
    city_attractiveness: tuple[CityAttractiveness, ...] = ()
    fuzzy_search: bool = False

    def __post_init__(self):
        identifier(self.departure_city)
        if type(self.fuzzy_search) is not bool:
            raise ValueError("fuzzy_search must be a boolean")
        if self.max_trip_duration is not None and (
            type(self.max_trip_duration) is not int
            or not 1 <= self.max_trip_duration <= 2**31 - 1
        ):
            raise ValueError("max_trip_duration must be positive GraphQL Int seconds")
        weighted_cities = [item.city for item in self.city_attractiveness]
        if len(set(weighted_cities)) != len(weighted_cities):
            raise ValueError("attractiveness cities must be distinct")
        if set(weighted_cities) & {self.departure_city, *self.excluded_cities}:
            raise ValueError("attractiveness cannot refer to home or excluded cities")
        for cities in (self.mandatory_cities, self.excluded_cities):
            for city in cities:
                identifier(city)
            if len(cities) != len(set(cities)):
                raise ValueError("city lists must contain distinct identifiers")
        if self.departure_city in (*self.mandatory_cities, *self.excluded_cities):
            raise ValueError("home must not be mandatory or excluded")
        if set(self.mandatory_cities) & set(self.excluded_cities):
            raise ValueError("mandatory and excluded cities must not overlap")
        if not self.transport_types or len(set(self.transport_types)) != len(
            self.transport_types
        ):
            raise ValueError("transport_types must be non-empty and distinct")
        if any(not isinstance(mode, TransportType) for mode in self.transport_types):
            raise ValueError("unsupported transport mode")
        if (
            type(self.time_tolerance) is not int
            or not 0 <= self.time_tolerance <= 2**31 - 1
        ):
            raise ValueError("time_tolerance must be a non-negative GraphQL Int")
        if instant(self.desired_arrival_date) <= instant(self.departure_date):
            raise ValueError("desired_arrival_date must be after departure_date")
        # Check arithmetic range at the boundary, before exploring any states.
        self.return_window

    @property
    def return_window(self) -> tuple[datetime, datetime]:
        try:
            target = instant(self.desired_arrival_date)
            tolerance = timedelta(minutes=self.time_tolerance)
            latest = target + tolerance
            if self.max_trip_duration is not None:
                latest = min(
                    latest,
                    instant(self.departure_date)
                    + timedelta(seconds=self.max_trip_duration),
                )
            return target - tolerance, latest
        except OverflowError as exc:
            raise ValueError("return window exceeds supported datetime range") from exc


@dataclass(frozen=True)
class TripLeg:
    origin: str
    destination: str
    departure_at: datetime
    arrival_at: datetime
    transport_type: TransportType

    def __post_init__(self):
        identifier(self.origin)
        identifier(self.destination)
        if not isinstance(self.transport_type, TransportType):
            raise ValueError("unsupported transport mode")
        seconds = (
            instant(self.arrival_at) - instant(self.departure_at)
        ).total_seconds()
        if seconds <= 0 or not seconds.is_integer() or seconds > 2**31 - 1:
            raise ValueError("leg duration must be positive whole GraphQL Int seconds")

    @property
    def time_in_travel(self) -> int:
        return int(
            (instant(self.arrival_at) - instant(self.departure_at)).total_seconds()
        )


@dataclass(frozen=True)
class JourneyOffer:
    """One independently purchasable ticket, possibly with multiple segments."""

    id: str
    path: tuple[TripLeg, ...]
    price: Money
    source: str
    synthetic: bool
    connection_id: str | None = None
    operator: str | None = None

    def __post_init__(self):
        if not self.id or not self.path or not self.source:
            raise ValueError("offer requires identity, source and a non-empty path")
        for previous, following in zip(self.path, self.path[1:]):
            if previous.destination != following.origin or instant(
                following.departure_at
            ) < instant(previous.arrival_at):
                raise ValueError("offer segments must be connected and chronological")


@dataclass(frozen=True)
class CityStay:
    city: str
    time_spent: float
    target_time: float
    attractiveness: float


@dataclass(frozen=True)
class RoundTrip:
    offers: tuple[JourneyOffer, ...]
    total_price: Money
    city_stays: tuple[CityStay, ...] = ()
    stay_fit: float | None = None

    @property
    def path(self) -> tuple[TripLeg, ...]:
        return tuple(leg for offer in self.offers for leg in offer.path)

## Search algorithm

The bounded search skips already visited cities by default (including connections
inside tickets), except for the final return home. Visits are tracked per itinerary.
The algorithm retains every result found unless a configured result limit is reached.
Search warnings distinguish a complete finite snapshot from truncated exploration.

In [ ]:
"""Bounded traversal of a finite schedule snapshot with optional fuzzy stay ranking.

No provider calls or shared weight mutations occur during exploration. Price-first
searches find the cheapest supplied offers only when exploration is not truncated.
"""

from collections import defaultdict
from dataclasses import dataclass
from decimal import Decimal, localcontext
from typing import Iterable
import logging

logger = logging.getLogger("notebook.round_trip_search")



@dataclass(frozen=True)
class SearchLimits:
    max_legs: int = 8
    max_states: int = 100_000
    max_results: int = 20

    def __post_init__(self):
        if any(
            type(value) is not int or value < 1
            for value in (self.max_legs, self.max_states, self.max_results)
        ):
            raise ValueError("search limits must be positive integers")


@dataclass(frozen=True)
class SearchResult:
    options: tuple[RoundTrip, ...]
    states_explored: int
    truncation_reasons: tuple[str, ...]
    results_limited: bool


def find_round_trips(
    settings: SearchSettings,
    offers: Iterable[JourneyOffer],
    limits: SearchLimits = SearchLimits(),
    *,
    allow_city_revisits: bool = False,
) -> SearchResult:
    """Skip repeated cities per itinerary, except the final return home.

    Internal ticket connections count as visits. Set allow_city_revisits=True
    explicitly to explore schedules that pass through a city more than once.
    """
    if type(allow_city_revisits) is not bool:
        raise ValueError("allow_city_revisits must be a boolean")
    logger.info(
        "Starting search home=%s mandatory=%s budget=%s %s",
        settings.departure_city,
        settings.mandatory_cities,
        settings.expenses_tolerance.amount,
        settings.expenses_tolerance.currency,
    )
    adjacency = defaultdict(list)
    identities = {}
    excluded = set(settings.excluded_cities)
    home = settings.departure_city
    earliest, latest = settings.return_window
    for offer in offers:
        logger.debug(
            "Inspect offer=%s route=%s price=%s %s",
            offer.id,
            " -> ".join(
                (offer.path[0].origin, *(leg.destination for leg in offer.path))
            ),
            offer.price.amount,
            offer.price.currency,
        )
        if offer.id in identities:
            if identities[offer.id] != offer:
                raise ValueError(f"conflicting offer identity: {offer.id}")
            logger.debug("Skip duplicate offer=%s", offer.id)
            continue
        identities[offer.id] = offer
        if offer.price.currency != settings.expenses_tolerance.currency:
            raise ValueError(
                "offer currency differs from budget; currency conversion is unsupported"
            )
        if any(
            leg.transport_type not in settings.transport_types
            or {leg.origin, leg.destination} & excluded
            for leg in offer.path
        ):
            logger.debug("Reject offer=%s: excluded city or disallowed mode", offer.id)
            continue
        # Home inside a purchase would violate the first-return termination rule.
        if any(leg.destination == home for leg in offer.path[:-1]):
            logger.debug("Reject offer=%s: returns home inside ticket", offer.id)
            continue
        if (
            instant(offer.path[0].departure_at) < instant(settings.departure_date)
            or instant(offer.path[-1].arrival_at) > latest
        ):
            logger.debug("Reject offer=%s: outside departure/return horizon", offer.id)
            continue
        adjacency[offer.path[0].origin].append(offer)
        logger.debug("Index offer=%s", offer.id)
    for outgoing in adjacency.values():
        outgoing.sort(
            key=lambda offer: (
                offer.price.amount,
                instant(offer.path[0].departure_at),
                offer.id,
            )
        )
    # Directed reachability prunes dead ends without assumptions about prices/time.
    reverse = defaultdict(set)
    for city, outgoing in adjacency.items():
        for offer in outgoing:
            reverse[offer.path[-1].destination].add(city)
    reachable, pending = {home}, [home]
    while pending:
        for city in reverse[pending.pop()]:
            if city not in reachable:
                reachable.add(city)
                pending.append(city)
    logger.debug("Cities with a directed route home: %s", sorted(reachable))
    stack = [(home, instant(settings.departure_date), (), Decimal(0), frozenset(), 0)]
    candidates = []
    reasons = set()
    states = 0
    results_limited = False
    while stack:
        if states >= limits.max_states:
            logger.info("Stop exploration: max_states=%s", limits.max_states)
            reasons.add("max_states")
            break
        city, ready, chosen, total, visited, legs = stack.pop()
        states += 1
        logger.debug(
            "Explore state=%s city=%s ready=%s total=%s legs=%s tickets=%s",
            states,
            city,
            ready.isoformat(),
            total,
            legs,
            tuple(offer.id for offer in chosen),
        )
        if chosen and city == home:
            if (
                visited - {home}
                and set(settings.mandatory_cities) <= visited
                and earliest <= ready <= latest
            ):
                candidate = RoundTrip(
                    chosen, Money(total, settings.expenses_tolerance.currency)
                )
                stays, fit = city_stays(candidate, settings)
                candidates.append(
                    RoundTrip(
                        candidate.offers,
                        candidate.total_price,
                        stays,
                        fit if settings.fuzzy_search else None,
                    )
                )
                logger.info(
                    "Found round trip price=%s tickets=%s",
                    total,
                    tuple(offer.id for offer in chosen),
                )
                # Bound retained results without stopping exploration.
                candidates.sort(key=_rank)
                if len(candidates) > limits.max_results:
                    dropped = candidates.pop()
                    logger.debug(
                        "Discard ranked result beyond max_results: %s",
                        tuple(offer.id for offer in dropped.offers),
                    )
                    results_limited = True
            else:
                logger.debug(
                    "Reject return: required visits or return window not satisfied"
                )
            continue
        for offer in reversed(adjacency[city]):
            logger.debug("Consider next ticket=%s from=%s", offer.id, city)
            destination = offer.path[-1].destination
            # Track physical arrivals, including connections within one purchase.
            # The current origin is already visited; leaving it is not a revisit.
            next_visited = set(visited) | {city}
            repeated_city = False
            for index, leg in enumerate(offer.path):
                final_return = leg.destination == home and index == len(offer.path) - 1
                if leg.destination in next_visited and not final_return:
                    repeated_city = True
                    break
                next_visited.add(leg.destination)
            if repeated_city and not allow_city_revisits:
                logger.debug("Skip ticket=%s: city already visited", offer.id)
                continue
            if (
                destination not in reachable
                or instant(offer.path[0].departure_at) < ready
            ):
                logger.debug(
                    "Skip ticket=%s: missed departure or no directed route home",
                    offer.id,
                )
                continue
            amount = _add_exact(total, offer.price.amount)
            if amount > settings.expenses_tolerance.amount:
                logger.debug(
                    "Skip ticket=%s: price=%s exceeds budget", offer.id, amount
                )
                continue  # These are fixed purchase prices, not historical estimates.
            if legs + len(offer.path) > limits.max_legs:
                logger.debug("Skip ticket=%s: max_legs=%s", offer.id, limits.max_legs)
                reasons.add("max_legs")
                continue
            logger.debug(
                "Expand ticket=%s destination=%s arrival=%s total=%s",
                offer.id,
                destination,
                offer.path[-1].arrival_at.isoformat(),
                amount,
            )
            stack.append(
                (
                    destination,
                    instant(offer.path[-1].arrival_at),
                    chosen + (offer,),
                    amount,
                    visited | {leg.destination for leg in offer.path},
                    legs + len(offer.path),
                )
            )
    logger.info(
        "Search finished states=%s results=%s truncation=%s results_limited=%s",
        states,
        len(candidates),
        sorted(reasons),
        results_limited,
    )
    return SearchResult(
        tuple(candidates), states, tuple(sorted(reasons)), results_limited
    )


def _rank(option: RoundTrip):
    return (
        -(option.stay_fit if option.stay_fit is not None else 0),
        option.total_price.amount,
        sum(leg.time_in_travel for leg in option.path),
        tuple(offer.id for offer in option.offers),
    )


def _add_exact(left: Decimal, right: Decimal) -> Decimal:
    # Align every digit, plus a carry, independently of ambient Decimal precision.
    with localcontext() as context:
        context.prec = (
            max(left.adjusted(), right.adjusted(), 0)
            - min(left.as_tuple().exponent, right.as_tuple().exponent, 0)
            + 2
        )
        return left + right


def city_stays(
    option: RoundTrip, settings: SearchSettings
) -> tuple[tuple[CityStay, ...], float]:
    """Soft weighted targets; observed gaps never change the supplied schedule.

    Aggregate revisits and internal ticket connections. These are elapsed city
    times, not a guarantee of usable sightseeing time or terminal access.
    """
    path = option.path
    if not path:
        return (), 0.0
    home = settings.departure_city
    spent = {leg.destination: 0.0 for leg in path if leg.destination != home}
    for previous, following in zip(path, path[1:]):
        if previous.destination != home:
            spent[previous.destination] += (
                instant(following.departure_at) - instant(previous.arrival_at)
            ).total_seconds()
    if not spent:
        return (), 0.0
    # Initial waiting at home is not city exploration time. Travel, including
    # all physical legs of bundled tickets, consumes the available stay budget.
    target_return = min(
        instant(settings.desired_arrival_date), settings.return_window[1]
    )
    available = max(
        0.0,
        (target_return - instant(path[0].departure_at)).total_seconds()
        - sum(leg.time_in_travel for leg in path),
    )
    weights = {item.city: item.weight for item in settings.city_attractiveness}
    # Scale before summation to avoid overflow for large valid weights.
    scale = max(weights.get(city, 1.0) for city in spent)
    normalized = {city: weights.get(city, 1.0) / scale for city in spent}
    total_weight = sum(normalized.values())
    stays = tuple(
        CityStay(
            city,
            seconds,
            available * normalized[city] / total_weight,
            weights.get(city, 1.0),
        )
        for city, seconds in spent.items()
    )
    difference = sum(abs(stay.time_spent - stay.target_time) for stay in stays)
    denominator = sum(spent.values()) + available
    fit = max(0.0, min(1.0, 1.0 - difference / denominator)) if denominator else 0.0
    return stays, fit

## Base graph and on-demand API mocks

The snapshot loader uses the repository envelope format and restricted pickle
globals, checks the NetworkX version, and caches the graph until its file changes.
Only load locally produced snapshots. The path control accepts `.pickle` or `.pickle.zst`.
Missing snapshots produce an actionable error; no invented topology is substituted.

The whole snapshot remains one directed multigraph. API requests cover all existing
endpoint pairs for enabled modes; responses preserve every parallel connection, its key,
and operator. Neither a missing direction nor an unsupported mode is generated.
Reference travel times (or distance/speed when absent) seed the explicitly mock
timetable; fares and departure availability are never claimed to come from the graph.

Mocks expose `search(MockFareRequest) → MockFareResponse`. The same request and
seed reproduce the same tickets. Requests are collected before DFS and validated.
Generation is bounded to 14 days, while exploration has its own state/result limits.


In [ ]:
from pathlib import Path
from functools import lru_cache
from compression import zstd
import pickle
import networkx as nx

# Match the repository's local snapshot format, without a backend-package import.
_ALLOWED_SNAPSHOT_GLOBALS = {
    ("networkx.classes.multidigraph", "MultiDiGraph"),
    ("networkx.classes.function", "frozen"),
    *(("networkx.classes.coreviews", name) for name in
      ("AtlasView", "AdjacencyView", "MultiAdjacencyView")),
    *(("networkx.classes.reportviews", name) for name in
      ("NodeView", "NodeDataView", "OutMultiEdgeView", "InMultiEdgeView",
       "OutMultiEdgeDataView", "InMultiEdgeDataView", "DiMultiDegreeView",
       "InMultiDegreeView", "OutMultiDegreeView")),
}


class _GraphUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) not in _ALLOWED_SNAPSHOT_GLOBALS:
            raise pickle.UnpicklingError(f"Unsupported snapshot object: {module}.{name}")
        return super().find_class(module, name)


def find_base_graph():
    for root in (Path.cwd(), *Path.cwd().parents):
        for relative in ("backend/src/backend/infrastructure/fixtures/data/base_graph.pickle",
                         "src/backend/infrastructure/fixtures/data/base_graph.pickle"):
            path = root / relative
            if path.is_file():
                return str(path)
    return str(Path.cwd() / "backend/src/backend/infrastructure/fixtures/data/base_graph.pickle")


class BaseGraphSource:
    def __init__(self, graph, label="in-memory base graph"):
        if type(graph) is not nx.MultiDiGraph:
            raise ValueError("Base graph must be a directed NetworkX MultiDiGraph.")
        if any(not isinstance(city, str) for city in graph):
            raise ValueError("Base graph city IDs must be strings.")
        self.graph = graph
        self.label = label
        self.names = {
            city: f"{attrs.get('name', city)} ({attrs.get('country', 'unknown')})"
            for city, attrs in graph.nodes(data=True)
        }
        self.default_cities = []
        for names, country in (({"vilnius"}, "Lithuania"), ({"warsaw"}, "Poland"),
                               ({"krakow", "cracow"}, "Poland")):
            match = next((city for city, attrs in graph.nodes(data=True)
                          if str(attrs.get("name", "")).casefold() in names
                          and attrs.get("country") == country), None)
            if match is not None:
                self.default_cities.append(match)
        if not self.default_cities:
            self.default_cities = sorted(graph)[:3]
        self.default_cities = tuple(self.default_cities)

    def choices(self, query="", selected=(), limit=100):
        query = query.strip().casefold()
        matches = sorted((city for city in self.graph
                          if query in self.names[city].casefold() or query in city.casefold()),
                         key=lambda city: (self.names[city], city))[:limit]
        cities = sorted(set(matches) | set(selected), key=lambda city: (self.names[city], city))
        return [(self.names[city], city) for city in cities]

    def connections(self, origin, destination, mode):
        return [(key, attrs) for key, attrs in
                self.graph.get_edge_data(origin, destination, default={}).items()
                if attrs.get("mode") == mode.value]


@lru_cache(maxsize=1)
def _read_base_graph(path, modified_ns, size):
    # Only load locally produced snapshots. Do not deserialize downloaded pickle files.
    try:
        stream = zstd.open(path, "rb") if path.endswith(".pickle.zst") else open(path, "rb")
        with stream:
            envelope = _GraphUnpickler(stream).load()
    except (pickle.UnpicklingError, EOFError, zstd.ZstdError, AttributeError, TypeError) as exc:
        raise ValueError("Invalid base graph snapshot; regenerate it with just build-base-graph.") from exc
    if (not isinstance(envelope, dict) or envelope.get("format") != "explorer.base_graph"
        or envelope.get("format_version") != 1):
        raise ValueError("Unsupported base graph snapshot format.")
    if envelope.get("networkx_version") != nx.__version__:
        raise ValueError("Base graph NetworkX version mismatch; regenerate the snapshot.")
    if type(envelope.get("graph")) is not nx.MultiDiGraph:
        raise ValueError("Base graph snapshot must contain a directed MultiDiGraph.")
    return BaseGraphSource(nx.freeze(envelope["graph"]), path)


def load_graph_source(path):
    path = Path(path).expanduser().resolve()
    if not (path.name.endswith(".pickle") or path.name.endswith(".pickle.zst")):
        raise ValueError("Select a local .pickle or .pickle.zst base graph snapshot.")
    if not path.is_file():
        raise ValueError("Base graph snapshot not found. Generate it with just build-base-graph, then set its path.")
    stat = path.stat()
    return _read_base_graph(str(path), stat.st_mtime_ns, stat.st_size)


from hashlib import sha256
import random

@dataclass(frozen=True)
class MockFareRequest:
    origin: str
    destination: str
    earliest_departure: datetime
    latest_arrival: datetime
    seed: int = 42
    services_per_day: int = 1

    def __post_init__(self):
        if (not isinstance(self.origin, str) or not isinstance(self.destination, str)
            or not self.origin.strip() or not self.destination.strip()):
            raise ValueError("Mock API requires non-empty city IDs from the base graph.")
        if self.origin == self.destination:
            raise ValueError("Mock API origin and destination must differ.")
        if not isinstance(self.earliest_departure, datetime) or not isinstance(self.latest_arrival, datetime):
            raise ValueError("Mock API dates must be timezone-aware datetimes.")
        start, end = instant(self.earliest_departure), instant(self.latest_arrival)
        if not 0 < (end - start).total_seconds() <= 14 * 86400:
            raise ValueError("Mock API search window must be positive and at most 14 days.")
        if type(self.seed) is not int or not 0 <= self.seed <= 2**32-1:
            raise ValueError("Mock API seed must be an integer from 0 to 4294967295.")
        if type(self.services_per_day) is not int or not 1 <= self.services_per_day <= 3:
            raise ValueError("Mock API services per day must be an integer from 1 to 3.")


@dataclass(frozen=True)
class MockFareResponse:
    provider: str
    offers: tuple[JourneyOffer, ...]
    synthetic: bool = True


class MockTransportAPI:
    """In-process search API mock, not a server or a real provider contract.

    Fixed timetable rules plus request-local randomness generate physical tickets.
    Budget, ranking and query order never influence an individual service's fare.
    """
    mode = None
    source = ""
    minimum_cents = 0
    maximum_cents = 0

    def __init__(self, source):
        self.source_graph = source

    def search(self, request: MockFareRequest) -> MockFareResponse:
        if not isinstance(request, MockFareRequest):
            raise ValueError("Mock API search expects a MockFareRequest.")
        if self.mode is None:
            raise ValueError("Choose a concrete mock transport API.")
        start, end = instant(request.earliest_departure), instant(request.latest_arrival)
        offers = []
        arrival_hours = {1: (16,), 2: (10, 16), 3: (10, 16, 22)}[request.services_per_day]
        for connection_id, attrs in self.source_graph.connections(request.origin, request.destination, self.mode):
            day = start.date()
            while day <= end.date():
                for hour in arrival_hours:
                    identity = f"{request.seed}|{self.source}|{request.origin}|{request.destination}|{connection_id}|{day}|{hour}"
                    digest = sha256(identity.encode()).hexdigest()
                    rng = random.Random(digest)
                    reference_minutes = attrs.get("travel_time_minutes")
                    distance = attrs.get("distance_km")
                    if isinstance(reference_minutes, (int, float)) and math.isfinite(reference_minutes) and reference_minutes > 0:
                        minutes = max(1, round(reference_minutes))
                    elif isinstance(distance, (int, float)) and math.isfinite(distance) and distance > 0:
                        speed = {Mode.PLANES: 700, Mode.TRAINS: 100, Mode.BUSSES: 60, Mode.FERRIES: 30}[self.mode]
                        minutes = max(1, round(distance / speed * 60))
                    else:
                        minutes = {Mode.PLANES: 60, Mode.TRAINS: 180, Mode.BUSSES: 240, Mode.FERRIES: 360}[self.mode]
                    # Reference durations are inputs to an explicitly mock timetable.
                    minutes += rng.randrange(4) * 15
                    arrival = datetime.combine(day, datetime.min.time(), timezone.utc) + timedelta(hours=hour)
                    departure = arrival - timedelta(minutes=minutes)
                    cents = rng.randint(self.minimum_cents, self.maximum_cents)
                    if departure < start or arrival > end:
                        continue
                    leg = TripLeg(request.origin, request.destination, departure, arrival, self.mode)
                    offers.append(JourneyOffer(
                        "mock:" + digest[:24], (leg,), Money(Decimal(cents) / 100, "USD"),
                        self.source, True, str(connection_id), attrs.get("operator"),
                    ))
                day += timedelta(days=1)
        return MockFareResponse(self.source, tuple(offers))


class MockFlightAPI(MockTransportAPI):
    mode, source, minimum_cents, maximum_cents = Mode.PLANES, "mock-flight-api", 4500, 9000


class MockTrainAPI(MockTransportAPI):
    mode, source, minimum_cents, maximum_cents = Mode.TRAINS, "mock-train-api", 1800, 3800


class MockBusAPI(MockTransportAPI):
    mode, source, minimum_cents, maximum_cents = Mode.BUSSES, "mock-bus-api", 1200, 2500


class MockFerryAPI(MockTransportAPI):
    mode, source, minimum_cents, maximum_cents = Mode.FERRIES, "mock-ferry-api", 2500, 4500


def collect_mock_offers(settings, *, source, seed, services_per_day, providers=None):
    """Query every existing connection in the whole graph before route search."""
    if settings.departure_city not in source.names:
        raise ValueError("Choose a home city from the base graph.")
    if settings.expenses_tolerance.currency != "USD":
        raise ValueError("Mock APIs support USD only.")
    providers = providers if providers is not None else (
        MockFlightAPI(source), MockTrainAPI(source), MockBusAPI(source), MockFerryAPI(source),
    )
    offers, calls = [], []
    latest = settings.return_window[1]
    if latest <= instant(settings.departure_date):
        return (), ()
    for provider in providers:
        if provider.mode not in settings.transport_types:
            continue
        # Group existing directed connections by endpoint pair; each API response
        # preserves all parallel edges for its mode. Never infer reverse edges.
        pairs = sorted({(origin, destination) for origin, destination, attrs in
                        source.graph.edges(data=True)
                        if origin != destination
                        and attrs.get("mode") == provider.mode.value})
        for origin, destination in pairs:
            request = MockFareRequest(origin, destination, settings.departure_date,
                                      latest, seed, services_per_day)
            response = provider.search(request)
            if (not isinstance(response, MockFareResponse) or not response.synthetic
                or response.provider != provider.source):
                raise ValueError("Mock API returned an invalid synthetic response.")
            connection_ids = {str(key) for key, _ in source.connections(origin, destination, provider.mode)}
            for offer in response.offers:
                if (not isinstance(offer, JourneyOffer) or not offer.synthetic
                    or offer.price.currency != "USD"
                    or offer.source != response.provider
                    or offer.path[0].origin != origin
                    or offer.path[-1].destination != destination
                    or instant(offer.path[0].departure_at) < instant(request.earliest_departure)
                    or instant(offer.path[-1].arrival_at) > instant(request.latest_arrival)
                    or offer.connection_id not in connection_ids
                    or any(leg.transport_type != provider.mode for leg in offer.path)):
                    raise ValueError("Mock API returned an offer outside its request contract.")
            offers.extend(response.offers)
            calls.append((provider.source, request))
    return tuple(offers), tuple(calls)


## Itinerary tables

Each option shows all travel legs and city stops in UTC. Gaps are elapsed time,
not guaranteed sightseeing time. Ticket fares are counted once, including bundles.

In [ ]:
def duration(seconds):
    seconds = int(seconds)
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours}h {minutes:02d}m" + (f" {seconds:02d}s" if seconds else "")


def timestamp(value):
    return instant(value).strftime("%Y-%m-%d %H:%M:%S UTC")


def itinerary_table(option, city_names=None):
    """Chronological physical legs and city gaps; charge each purchase once."""
    city_names = city_names or {}
    rows = []
    previous = None
    for ticket in option.offers:
        for index, leg in enumerate(ticket.path):
            if previous is not None:
                wait = (instant(leg.departure_at) - instant(previous.arrival_at)).total_seconds()
                rows.append({
                    "Activity": "City stop / interchange", "From / city": city_names.get(previous.destination, previous.destination),
                    "To": "", "Mode": "", "Start (UTC)": timestamp(previous.arrival_at),
                    "End (UTC)": timestamp(leg.departure_at), "Time spent": duration(wait),
                    "Ticket": "", "Fare charged": "", "Base connection": "", "Operator": "",
                })
            rows.append({
                "Activity": "Travel", "From / city": city_names.get(leg.origin, leg.origin), "To": city_names.get(leg.destination, leg.destination),
                "Mode": leg.transport_type.value, "Start (UTC)": timestamp(leg.departure_at),
                "End (UTC)": timestamp(leg.arrival_at),
                "Time spent": duration(leg.time_in_travel), "Ticket": ticket.id,
                "Base connection": ticket.connection_id or "", "Operator": ticket.operator or "",
                "Fare charged": (f"{ticket.price.currency} {ticket.price.amount}"
                                 if index == 0 else "included in ticket"),
            })
            previous = leg
    return pd.DataFrame(rows)


def table_html(frame):
    # No pandas row/column display limits: every leg and interchange is visible.
    return '<div style="overflow-x:auto">' + frame.to_html(index=False, escape=True, border=0) + '</div>'


def opportunities_html(search_result, search_settings, city_names=None):
    city_names = city_names or {}
    budget = search_settings.expenses_tolerance
    incomplete = bool(search_result.truncation_reasons or search_result.results_limited)
    coverage = (
        "Incomplete search: more options may exist. Limits reached: "
        + ", ".join((*search_result.truncation_reasons,
                     *(("max_results",) if search_result.results_limited else ())))
        if incomplete else "All feasible options in the supplied synthetic snapshot."
    )
    parts = [
        "<h2>Travel opportunities</h2>",
        f"<p>Budget: {escape(budget.currency)} {escape(str(budget.amount))}. "
        f"{len(search_result.options)} options; {search_result.states_explored} explored states.</p>",
        f"<p><strong>{escape(coverage)}</strong></p>",
    ]
    if not search_result.options:
        parts.append("<p>No feasible round trips found under the current budget and constraints.</p>")
    for number, option in enumerate(search_result.options, 1):
        path = option.path
        movement = sum(leg.time_in_travel for leg in path)
        elapsed = (instant(path[-1].arrival_at) - instant(path[0].departure_at)).total_seconds()
        route = " → ".join(city_names.get(city, city) for city in [path[0].origin, *(leg.destination for leg in path)])
        parts.extend([
            f"<h3>Option {number} · {escape(option.total_price.currency)} "
            f"{escape(str(option.total_price.amount))}</h3>",
            f"<p>{escape(route)}</p>",
            f"<p>Total trip: {duration(elapsed)} · Travel: {duration(movement)} · "
            f"Time in cities: {duration(elapsed - movement)} · "
            f"{len(path) - 1} interchanges / city stops</p>",
            table_html(itinerary_table(option, city_names)),
        ])
    return "\n".join(parts)

## Reactive configuration

Choose the base graph snapshot and home city. The entire snapshot is always
used as one directed multigraph: no city selection, region, or connected-component
partition limits the mock API requests or possible interchange cities.
**Find graph cities** filters only the home, required, excluded and preference
selectors (first 100 matches plus their current selections). It does not filter
connections or the search network.

Defaults use Vilnius as home and Cracow/Krakow as required when present.
Change dates, modes, seed or daily service count to generate mock tickets on all
existing graph connections for the enabled modes. **Generate new sample** advances
the seed. Repeated cities remain disabled by default, except the final return home.
Invalid inputs clear old results until corrected.

Zero leg/result limits retain all supplied legs and all candidates found within
the state budget. Warnings identify incomplete searches; the state budget can
truncate exploration of the full graph. Exact decimals use USD.
Date pickers use browser time; tables use UTC. Mock services arrive at 16:00 UTC
(or 10:00/16:00/22:00 with more services). City weights are positive preferences.


In [ ]:
class ReactiveTripDemo:
    """Own the widgets and recompute the complete report on every committed edit."""

    def __init__(self, graph_source=None):
        self.injected_source = graph_source
        self.graph_source = graph_source
        cities = [(graph_source.names[city], city) for city in graph_source.default_cities] if graph_source else []
        defaults = tuple(city for _, city in cities)
        home = defaults[0] if defaults else None
        required = defaults[-1:] if len(defaults) > 1 else ()
        style = {"description_width": "initial"}
        def control(kind, **kwargs):
            return kind(style=style, layout=widgets.Layout(width="min(100%, 420px)", margin="6px"), **kwargs)

        self.controls = {
            "home": control(widgets.Dropdown, options=cities, value=home, description="Home city"),
            "departure": control(widgets.DatetimePicker, value=datetime(2026, 12, 1, tzinfo=timezone.utc), description="Earliest departure"),
            "return": control(widgets.DatetimePicker, value=datetime(2026, 12, 3, 16, tzinfo=timezone.utc), description="Target return"),
            "tolerance": control(widgets.BoundedIntText, value=0, min=0, max=2**31-1, description="Return tolerance (minutes)"),
            "budget": control(widgets.Text, value="200.00", description="Budget (USD)", continuous_update=False),
            "modes": control(widgets.SelectMultiple, options=[(m.value, m) for m in Mode], value=(Mode.BUSSES, Mode.TRAINS, Mode.PLANES), description="Transport modes"),
            "required": control(widgets.SelectMultiple, options=cities, value=required, description="Required cities"),
            "excluded": control(widgets.SelectMultiple, options=cities, value=(), description="Excluded cities"),
            "revisits": control(widgets.Checkbox, value=False, description="Allow city revisits"),
            "max_duration": control(widgets.BoundedIntText, value=0, min=0, max=2**31-1, description="Max trip seconds (0 = off)"),
            "fuzzy": control(widgets.Checkbox, value=False, description="Rank by city-stay fit"),
            "weighted_cities": control(widgets.SelectMultiple, options=cities, value=(), description="Cities with custom weights"),

            "max_states": control(widgets.BoundedIntText, value=100_000, min=1, max=2**31-1, description="Maximum search states"),
            "max_legs": control(widgets.BoundedIntText, value=0, min=0, max=2**31-1, description="Max legs (0 = all supplied)"),
            "max_results": control(widgets.BoundedIntText, value=0, min=0, max=2**31-1, description="Max results (0 = all found)"),
        }
        self.controls["graph_path"] = control(
            widgets.Text, value=find_base_graph(), description="Base graph snapshot", continuous_update=False,
        )
        self.controls["city_filter"] = control(
            widgets.Text, value="", description="Find graph cities (first 100)", continuous_update=False,
        )
        self.controls["seed"] = control(
            widgets.BoundedIntText, value=42, min=0, max=2**32-1, description="Mock API seed",
        )
        self.controls["services"] = control(
            widgets.BoundedIntText, value=1, min=1, max=3, description="Services / route / day / mode",
        )
        self.generate = widgets.Button(description="Generate new sample", icon="refresh")
        self.generate.on_click(self.generate_sample)
        for name, city in cities:
            self.controls["weight:" + city] = control(
                widgets.FloatText, value=1.0, description=f"{name} relative weight",
            )
        self.status = widgets.HTML()
        self.report = widgets.HTML()
        self.help = widgets.HTML(value=(
            "<p>In-process API mocks generate synthetic schedules and USD fares for the selected "
            "dates across the entire base graph. Only existing directed connections/modes are used; "
            "parallel edges stay separate. Schedules and fares are synthetic, not real availability. "
            "The window is limited to 14 days. City search filters selectors only; all graph cities can be interchanges.</p>"
            "<p>Changing a widget queries the mocks and refreshes the report. Generate new sample "
            "advances the seed. The same seed and request reproduce the same tickets. Services arrive "
            "at 16:00 UTC daily (or 10:00/16:00/22:00 with more services). "
            "Date pickers use browser time; tables use UTC. Commit text edits with Enter or blur. "
            "Use Ctrl/Cmd-click for multiple selections. Unselected city weights default to 1.</p>"
        ))
        self.fields = widgets.Box(
            children=list(self.controls.values()),
            layout=widgets.Layout(display="flex", flex_flow="row wrap"),
        )
        self.view = widgets.VBox([self.help, self.fields, self.generate, self.status, self.report])
        self.settings = self.offers = self.limits = self.result = None
        self.report_html = ""
        self.api_calls = ()
        self.error = None
        self.closed = False
        self.refreshing = False
        for widget in self.controls.values():
            widget.observe(self.refresh, names="value")
        self.refresh()

    def refresh(self, change=None):
        if self.closed or self.refreshing:
            return
        self.refreshing = True
        self.status.value = '<p role="status">Searching synthetic schedules…</p>'
        self.result = None
        self.settings = self.offers = self.limits = None
        self.api_calls = ()
        self.error = None
        self.report.value = self.report_html = ""
        value = lambda key: self.controls[key].value
        try:
            source = self.injected_source or load_graph_source(value("graph_path"))
            if source is not self.graph_source:
                self.graph_source = source
                defaults = source.default_cities
                for key in ("home", "required", "excluded", "weighted_cities"):
                    self.controls[key].options = [(source.names[city], city) for city in defaults]
                    self.controls[key].value = (defaults[0] if defaults else None) if key == "home" else (defaults[-1:] if key == "required" and len(defaults) > 1 else ())
            for key in ("home", "required", "excluded", "weighted_cities"):
                current = value(key)
                selected = (current,) if key == "home" and current else current or ()
                self.controls[key].options = source.choices(value("city_filter"), selected)
                self.controls[key].value = current
            # Create weight inputs only for explicit preferences, not every graph city.
            for city in value("weighted_cities"):
                if "weight:" + city not in self.controls:
                    widget = widgets.FloatText(value=1.0, description=source.names[city] + " weight",
                        style={"description_width": "initial"}, layout=widgets.Layout(width="min(100%, 420px)", margin="6px"))
                    self.controls["weight:" + city] = widget
                    widget.observe(self.refresh, names="value")
                    self.fields.children = (*self.fields.children, widget)
            if value("home") is None:
                raise ValueError("Base graph has no cities. Select another snapshot.")
            if value("departure") is None or value("return") is None:
                raise ValueError("Choose both departure and target return dates.")
            budget = Money(Decimal(value("budget")), "USD")
            self.settings = SearchSettings(
                departure_city=value("home"), departure_date=value("departure"),
                desired_arrival_date=value("return"), time_tolerance=value("tolerance"),
                expenses_tolerance=budget, transport_types=tuple(value("modes")),
                mandatory_cities=tuple(value("required")), excluded_cities=tuple(value("excluded")),
                max_trip_duration=value("max_duration") or None,
                fuzzy_search=value("fuzzy"),
                city_attractiveness=tuple(
                    CityAttractiveness(city, value("weight:" + city))
                    for city in value("weighted_cities")
                ),
            )
            self.offers, self.api_calls = collect_mock_offers(
                self.settings, source=source, seed=value("seed"), services_per_day=value("services"),
            )
            self.limits = SearchLimits(
                max_legs=value("max_legs") or max(1, sum(len(o.path) for o in self.offers)),
                max_states=value("max_states"),
                max_results=value("max_results") or value("max_states"),
            )
            self.result = find_round_trips(
                self.settings, self.offers, self.limits, allow_city_revisits=value("revisits"),
            )
            self.report_html = opportunities_html(self.result, self.settings, source.names)
            self.report.value = self.report_html
            self.status.value = (
                f'<p role="status">Generated {len(self.offers)} synthetic tickets with '
                f'{len(self.api_calls)} mock API requests; seed {value("seed")}. '
                f'Base graph: {source.graph.number_of_nodes()} cities / {source.graph.number_of_edges()} connections; '
                f'whole graph used without city or regional partitions.</p>'
            )
        except (ValueError, InvalidOperation, OverflowError, TimeoutError, OSError) as exc:
            self.error = "Enter a valid decimal USD budget." if isinstance(exc, InvalidOperation) else str(exc)
            self.result = None
            self.status.value = f'<p role="alert">Check configuration: {escape(self.error)}</p>'
        finally:
            self.refreshing = False

    def generate_sample(self, button=None):
        if not self.closed:
            seed = self.controls["seed"]
            seed.value = (seed.value + 1) % (2**32)

    def close(self):
        if self.closed:
            return
        self.closed = True
        for widget in self.controls.values():
            widget.unobserve(self.refresh, names="value")
            widget.close()
        self.generate.on_click(self.generate_sample, remove=True)
        for widget in (self.help, self.status, self.report, self.fields, self.generate, self.view):
            widget.close()


# Rerunning this cell detaches observers from the previous panel.
if "demo" in globals():
    demo.close()
demo = ReactiveTripDemo()
display(demo.view)